# Motor de Next-Best-Offer Hiperpersonalizado

## Objetivos de Aprendizaje

En este notebook, aprenderás a:

1. **Predecir propensión de compra** por producto con `SNOWFLAKE.ML.CLASSIFICATION`
2. **Construir features de comportamiento** multi-producto
3. **Generar recomendaciones personalizadas** con `CORTEX.COMPLETE()`
4. **Optimizar next-best-offer** con scoring multi-modelo
5. **Crear dashboard de oportunidades** comerciales

---

## Lo Que Construirás

Un **motor de recomendación bancaria** que sugiere el mejor producto:
- Modelos de propensión por categoría (inversión, seguros, préstamos)
- Features de afinidad y momento del ciclo de vida
- Recomendaciones con explicación IA
- Dashboard para gestores comerciales

**Valor de Negocio**: Incrementar cross-selling y share-of-wallet con ofertas relevantes.

---

## Funcionalidades Clave

- `SNOWFLAKE.ML.CLASSIFICATION` — Modelos de propensión
- `CORTEX.COMPLETE()` — Argumentarios de venta personalizados
- `NTILE()` — Segmentación por deciles de propensión
- Window functions — Patrones de comportamiento

¡Comencemos!

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS NBO_MOTOR_DB;
CREATE SCHEMA IF NOT EXISTS NBO_MOTOR_DB.PUBLIC;
USE SCHEMA NBO_MOTOR_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() AS db;

---

## Paso 2: Generar Datos de Clientes y Productos

### Tablas

- **3.000 clientes** con perfil financiero y productos actuales
- **Catálogo de 15 productos** bancarios
- **Historial de contrataciones** recientes

In [ ]:
-- Clientes con perfil
CREATE OR REPLACE TABLE CLIENTES_NBO (
    cliente_id VARCHAR(10) PRIMARY KEY,
    edad INTEGER,
    segmento VARCHAR(20),
    ingresos_mensuales DECIMAL(10,2),
    saldo_medio DECIMAL(12,2),
    num_productos_actual INTEGER,
    tiene_cuenta_ahorro BOOLEAN,
    tiene_hipoteca BOOLEAN,
    tiene_inversion BOOLEAN,
    tiene_seguro BOOLEAN,
    meses_desde_ultima_contratacion INTEGER,
    contrato_producto_inversion BOOLEAN
);

INSERT INTO CLIENTES_NBO
SELECT
    'CLI' || LPAD(SEQ4()::STRING, 5, '0'),
    UNIFORM(22, 70, RANDOM()),
    ARRAY_CONSTRUCT('Premium','Particular','Joven','Senior')[UNIFORM(0,3,RANDOM())]::VARCHAR,
    ROUND(UNIFORM(1200, 12000, RANDOM()), 2),
    ROUND(UNIFORM(1000, 300000, RANDOM()), 2),
    UNIFORM(1, 6, RANDOM()),
    UNIFORM(0,1,RANDOM()) < 0.8,
    UNIFORM(0,1,RANDOM()) < 0.3,
    UNIFORM(0,1,RANDOM()) < 0.25,
    UNIFORM(0,1,RANDOM()) < 0.2,
    UNIFORM(1, 36, RANDOM()),
    CASE WHEN UNIFORM(0,100,RANDOM()) < 25 THEN TRUE ELSE FALSE END
FROM TABLE(GENERATOR(ROWCOUNT => 3000));

SELECT segmento, COUNT(*), ROUND(AVG(ingresos_mensuales),0) AS ing_medio
FROM CLIENTES_NBO GROUP BY 1;

---

## Paso 3: Ingeniería de Variables de Propensión

### Features para Next-Best-Offer

- **Capacidad financiera**: Ratio ahorro/ingreso, saldo disponible
- **Momento de vida**: Edad, segmento, tiempo desde última contratación
- **Cross-sell potencial**: Productos que NO tiene pero su perfil indica afinidad
- **Engagement**: Frecuencia de uso de canales

In [ ]:
-- Features de propensión
CREATE OR REPLACE TABLE FEATURES_NBO AS
SELECT
    cliente_id,
    edad::FLOAT,
    ingresos_mensuales::FLOAT,
    saldo_medio::FLOAT,
    num_productos_actual::FLOAT,
    (saldo_medio / NULLIF(ingresos_mensuales, 0))::FLOAT AS ratio_ahorro_ingreso,
    tiene_cuenta_ahorro::FLOAT,
    tiene_hipoteca::FLOAT,
    tiene_inversion::FLOAT,
    tiene_seguro::FLOAT,
    meses_desde_ultima_contratacion::FLOAT,
    CASE WHEN segmento = 'Premium' THEN 3 WHEN segmento = 'Senior' THEN 2 WHEN segmento = 'Particular' THEN 1 ELSE 0 END::FLOAT AS nivel_segmento,
    contrato_producto_inversion
FROM CLIENTES_NBO;

SELECT * FROM FEATURES_NBO LIMIT 10;

---

## Paso 4: Entrenar Modelo de Propensión

### Modelo para Inversión

Predicimos la propensión a contratar un producto de inversión.

In [ ]:
CREATE OR REPLACE TABLE TRAIN_NBO AS SELECT * FROM FEATURES_NBO SAMPLE (80);
CREATE OR REPLACE TABLE TEST_NBO AS SELECT * FROM FEATURES_NBO WHERE cliente_id NOT IN (SELECT cliente_id FROM TRAIN_NBO);

CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION modelo_propension_inv(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'TRAIN_NBO'),
    TARGET_COLNAME => 'CONTRATO_PRODUCTO_INVERSION'
);

In [ ]:
CALL modelo_propension_inv!SHOW_EVALUATION_METRICS();
CALL modelo_propension_inv!SHOW_FEATURE_IMPORTANCE();

---

## Paso 5: Puntuar y Generar Ofertas

### Ranking de Oportunidades

Usamos `NTILE(10)` para crear deciles de propensión.

In [ ]:
-- Puntuar clientes
CREATE OR REPLACE TABLE OFERTAS_NBO AS
SELECT
    cliente_id, edad, ingresos_mensuales, saldo_medio, num_productos_actual,
    tiene_inversion, contrato_producto_inversion AS target_real,
    modelo_propension_inv!PREDICT(
        OBJECT_CONSTRUCT(
            'EDAD', edad, 'INGRESOS_MENSUALES', ingresos_mensuales,
            'SALDO_MEDIO', saldo_medio, 'NUM_PRODUCTOS_ACTUAL', num_productos_actual,
            'RATIO_AHORRO_INGRESO', ratio_ahorro_ingreso,
            'TIENE_CUENTA_AHORRO', tiene_cuenta_ahorro,
            'TIENE_HIPOTECA', tiene_hipoteca,
            'TIENE_INVERSION', tiene_inversion,
            'TIENE_SEGURO', tiene_seguro,
            'MESES_DESDE_ULTIMA_CONTRATACION', meses_desde_ultima_contratacion,
            'NIVEL_SEGMENTO', nivel_segmento
        )
    ) AS pred,
    ROUND(pred['probability']['true']::FLOAT * 100, 1) AS propension_pct,
    NTILE(10) OVER (ORDER BY pred['probability']['true']::FLOAT DESC) AS decil_propension
FROM TEST_NBO
WHERE tiene_inversion = 0;

SELECT decil_propension, COUNT(*) AS n, ROUND(AVG(propension_pct),1) AS prop_media
FROM OFERTAS_NBO GROUP BY 1 ORDER BY 1;

---

## Paso 6: Argumentarios IA Personalizados

Generamos argumentarios de venta con `CORTEX.COMPLETE()`.

In [ ]:
CREATE OR REPLACE TABLE ARGUMENTARIOS_NBO AS
SELECT
    cliente_id, propension_pct, saldo_medio, ingresos_mensuales, edad,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large',
        'Eres un asesor financiero. Genera un argumentario de venta de 3 líneas para ofrecer un producto de inversión:\n' ||
        'Cliente: edad ' || edad || ', ingresos ' || ingresos_mensuales || '€/mes, saldo ' || saldo_medio || '€\n' ||
        'Propensión: ' || propension_pct || '%\n' ||
        'Responde en español con tono profesional y cercano.'
    ) AS argumentario
FROM OFERTAS_NBO
WHERE decil_propension <= 2
SAMPLE (20 ROWS);

SELECT * FROM ARGUMENTARIOS_NBO LIMIT 5;

---

## Paso 7: Dashboard de Oportunidades

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title('Motor de Next-Best-Offer')
st.markdown('### Oportunidades de Inversión')

total = session.sql('SELECT COUNT(*) FROM OFERTAS_NBO').collect()[0][0]
top = session.sql('SELECT COUNT(*) FROM OFERTAS_NBO WHERE decil_propension <= 2').collect()[0][0]

c1, c2 = st.columns(2)
c1.metric('Clientes sin Inversión', f'{total:,}')
c2.metric('Top 20% Propensión', f'{top:,}')

st.markdown('---')
st.subheader('Distribución por Decil')
df = session.sql('SELECT decil_propension, COUNT(*) AS n FROM OFERTAS_NBO GROUP BY 1 ORDER BY 1').to_pandas()
st.bar_chart(df.set_index('DECIL_PROPENSION'))

st.markdown('---')
st.subheader('Top Oportunidades con Argumentario')
df_arg = session.sql('SELECT cliente_id, propension_pct, saldo_medio, argumentario FROM ARGUMENTARIOS_NBO ORDER BY propension_pct DESC LIMIT 15').to_pandas()
st.dataframe(df_arg)

---

## Paso 8: Limpieza (Opcional)

In [ ]:
-- Descomentar para limpiar

-- DROP DATABASE IF EXISTS NBO_MOTOR_DB;